# Data Classification Using AI
### Exploratory Data Analysis & Preprocessing — Wine Recognition Dataset

**Internship Project — AI Track**

This notebook performs exploratory data analysis (EDA) on the Wine Recognition
dataset (UCI Machine Learning Repository, bundled with scikit-learn) prior to
model training. The goal is to classify wine samples into one of **three
cultivars** based on **13 chemical/physical measurements** obtained via
chemical analysis.

**Contents**
1. Data loading & structure
2. Data quality checks (missing values, duplicates, types)
3. Descriptive statistics
4. Class distribution
5. Univariate analysis (feature distributions)
6. Bivariate / multivariate analysis (correlation, pairwise relationships)
7. Outlier inspection
8. Feature scaling demonstration
9. Key EDA takeaways for modeling


In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_raw_dataset, get_feature_and_target_info
from preprocessing import clean_data, split_features_target, train_test_splitter, scale_features

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 20)


## 1. Data Loading & Structure

In [ ]:
df = load_raw_dataset()
print(f"Shape: {df.shape}")
df.head()


In [ ]:
df.info()


In [ ]:
info = get_feature_and_target_info()
print(f"Samples : {info['n_samples']}")
print(f"Features: {info['n_features']}")
print(f"Classes : {info['n_classes']} -> {info['target_names']}")


## 2. Data Quality Checks

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")

df_clean = clean_data(df)


## 3. Descriptive Statistics

In [ ]:
df_clean.describe().T


## 4. Class Distribution

Checking whether the three wine cultivars are balanced — this determines whether we need stratified sampling and/or class-weighting during training.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

df_clean['target_name'].value_counts().plot(kind='bar', ax=ax[0], color=['#2563eb', '#f59e0b', '#16a34a'])
ax[0].set_title('Class Counts')
ax[0].set_xlabel('Cultivar')
ax[0].set_ylabel('Count')
ax[0].tick_params(axis='x', rotation=0)

df_clean['target_name'].value_counts().plot(kind='pie', ax=ax[1], autopct='%1.1f%%',
                                             colors=['#2563eb', '#f59e0b', '#16a34a'])
ax[1].set_title('Class Proportion')
ax[1].set_ylabel('')

plt.tight_layout()
plt.show()


**Observation:** The classes are moderately imbalanced (59 / 71 / 48 samples) but not severely so. We use a **stratified train/test split** to preserve these proportions in both sets.

## 5. Univariate Analysis — Feature Distributions

In [ ]:
feature_cols = [c for c in df_clean.columns if c not in ('target', 'target_name')]

fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    sns.histplot(data=df_clean, x=feat, hue='target_name', kde=True, ax=axes[i], legend=(i == 0))
    axes[i].set_title(feat, fontsize=9)
    axes[i].set_xlabel('')

for j in range(len(feature_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


**Observation:** Several features (e.g. `flavanoids`, `color_intensity`, `proline`, `od280/od315_of_diluted_wines`) show visibly separated distributions across the three cultivars — a strong early signal that they will be highly predictive.

## 6. Bivariate / Multivariate Analysis

In [ ]:
plt.figure(figsize=(11, 9))
corr = df_clean[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            annot_kws={'size': 7}, square=True, cbar_kws={'shrink': 0.7})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
key_features = ['flavanoids', 'color_intensity', 'proline', 'od280/od315_of_diluted_wines', 'target_name']
sns.pairplot(df_clean[key_features], hue='target_name', palette=['#2563eb', '#f59e0b', '#16a34a'], diag_kind='kde')
plt.suptitle('Pairwise Relationships — Top Discriminative Features', y=1.02, fontsize=13, fontweight='bold')
plt.show()


**Observation:** `flavanoids` vs `od280/od315_of_diluted_wines` and `flavanoids` vs `color_intensity` show visibly separable clusters per class — these will likely emerge as top-importance features in tree-based models.

## 7. Outlier Inspection

In [ ]:
plt.figure(figsize=(14, 6))
df_melt = df_clean[feature_cols].melt(var_name='feature', value_name='value')
# Normalize each feature to [0,1] just for visual comparability in one boxplot grid
df_norm = df_clean[feature_cols].apply(lambda x: (x - x.min()) / (x.max() - x.min()))
df_norm_melt = df_norm.melt(var_name='feature', value_name='normalized_value')
sns.boxplot(data=df_norm_melt, x='feature', y='normalized_value', hue='feature', palette='deep', legend=False)
plt.xticks(rotation=60, ha='right')
plt.title('Boxplots of Min-Max Normalized Features (Outlier Screening)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


**Observation:** A handful of mild outliers exist (e.g. in `magnesium`, `proanthocyanins`) but none are extreme enough to indicate data-entry errors. We retain them, since tree-based ensembles are robust to such outliers and removing real chemical measurements could discard valid information.

## 8. Feature Scaling Demonstration

Features are on very different scales (e.g. `proline` ranges up to ~1680, while `hue` ranges ~0.4–1.8). Distance-based and gradient-based models (KNN, SVM, Logistic Regression, Neural Networks) require standardization.

In [ ]:
X, y, _ = split_features_target(df_clean)
X_train, X_test, y_train, y_test = train_test_splitter(X, y)
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

comparison = pd.DataFrame({
    'before_scaling_mean': X_train.mean(),
    'before_scaling_std': X_train.std(),
    'after_scaling_mean': X_train_scaled.mean().round(3),
    'after_scaling_std': X_train_scaled.std().round(3),
})
comparison


## 9. Key EDA Takeaways for Modeling

1. **178 samples, 13 features, 3 classes** — a small, clean, well-structured tabular dataset with no missing values or duplicates.
2. **Moderate class imbalance** (59 / 71 / 48) → use stratified train/test splitting and macro-averaged metrics (precision, recall, F1) rather than relying on accuracy alone.
3. **Strong class separability** in several features (`flavanoids`, `color_intensity`, `proline`, `od280/od315_of_diluted_wines`) suggests that even simple linear models should perform well, while ensemble/kernel methods should push accuracy close to 100%.
4. **Feature scales vary by orders of magnitude** → standardization is mandatory before training distance-based and gradient-based models.
5. **Multicollinearity** exists between some features (e.g. `flavanoids` and `total_phenols`) — tree-based models handle this natively; regularized linear models (`C` tuning) mitigate it for Logistic Regression / SVM.

➡️ Proceed to `02_Model_Training_Evaluation.ipynb` (or run `python main.py` from the project root) for multi-model training, hyperparameter tuning, and evaluation.
